# Bonus - Part 2: Beyond word ids (bag-of-words, TF-IDF, hashing, BPE, embeddings)
This **optional** notebook extends `TextProcess` from HW#9 Part 2. The graded deliverable is `Module_9_2.ipynb`; here the same idea
("turn text into numbers") is pushed further with four more representations, all implemented in `src/hw9/text_extras.py` and unit-tested in `tests/test_text_extras.py`.

| Representation | Output | Reversible? | Fixed size? | Unseen words |
|---|---|---|---|---|
| `TextProcess` ids (HW#9) | sequence of ints | yes (lower-cased) | no | `<OOV>` |
| Bag-of-words / TF-IDF | one vector per document | no (order lost) | yes (vocab size) | ignored |
| Hashing trick | sequence of ints | **no** | yes (buckets) | hashed anyway |
| BPE subwords | sequence of ints | yes | no | split into known pieces |
| Embedding layer | dense vectors learned by a network | - | - | needs ids first |

In [1]:
import sys
sys.path.insert(0, "src")          # run this notebook from the repository root
import numpy as np, pandas as pd
from hw9.text_process import TextProcess
from hw9.text_extras import (TextVectorizer, hash_encode, BPE, make_toy_sentiment, train_embedding_classifier)

corpus = [
    "The cat sat on the mat.",
    "The dog chased the cat!",
    "Dogs and cats don't always get along.",
    "Is the dog happy? Yes, the dog is very happy.",
    "Machine learning turns text into numbers; numbers feed neural networks.",
]

## 1. Bag-of-words and TF-IDF
A document becomes **one fixed-length vector** with one column per vocabulary term:
* *count* - how often the term occurs,
* *TF-IDF* - the count multiplied by an inverse document frequency (rare terms weigh more), then L2-normalised.

Word order is lost, so the text cannot be reconstructed from these vectors.

In [2]:
cnt = TextVectorizer("count").fit(corpus)
tfidf = TextVectorizer("tfidf").fit(corpus)
names = cnt.feature_names()
print("vocabulary size:", len(names), "| matrix shape:", cnt.transform(corpus).shape)

df_cnt = pd.DataFrame(cnt.transform(corpus), columns=names).astype(int)
df_cnt.index = [f"doc{i}" for i in range(len(corpus))]
df_cnt.loc[:, ["the", "cat", "dog", "happy", "numbers"]]

vocabulary size: 27 | matrix shape: (5, 27)


,the,cat,dog,happy,numbers
doc0,2,1,0,0,0
doc1,2,1,1,0,0
doc2,0,0,0,0,0
doc3,2,0,2,2,0
doc4,0,0,0,0,2


In [3]:
df_tfidf = pd.DataFrame(tfidf.transform(corpus), columns=names).round(3)
df_tfidf.index = df_cnt.index
print("'the' appears in most documents  -> idf =", round(float(tfidf.idf_[names.index('the')]), 3))
print("'numbers' appears in one document -> idf =", round(float(tfidf.idf_[names.index('numbers')]), 3))
print("every row has unit length:", np.allclose(np.linalg.norm(tfidf.transform(corpus), axis=1), 1))
df_tfidf.loc[:, ["the", "cat", "dog", "happy", "numbers"]]

'the' appears in most documents  -> idf = 1.405
'numbers' appears in one document -> idf = 2.099
every row has unit length: True


,the,cat,dog,happy,numbers
doc0,0.574,0.346,0.000,0.000,0.000
doc1,0.662,0.399,0.399,0.000,0.000
doc2,0.000,0.000,0.000,0.000,0.000
doc3,0.353,0.000,0.425,0.527,0.000
doc4,0.000,0.000,0.000,0.000,0.577


In [4]:
# n-grams keep a little word order: bigrams such as "the dog" become features of their own
bi = TextVectorizer("count", ngram_range=(1, 2)).fit(corpus)
print("unigram vocabulary:", len(names), "-> uni+bigram vocabulary:", len(bi.vocabulary_))
print([t for t in bi.feature_names() if " " in t][:10])

unigram vocabulary: 27 -> uni+bigram vocabulary: 57
['always get', 'and cats', 'cat sat', "cats don't", 'chased the', 'dog chased', 'dog happy', 'dog is', 'dogs and', "don't always"]


## 2. The hashing trick
`hash_encode` maps every token to `md5(token) % buckets`. **No vocabulary is stored** and unseen words never fail, but
(a) different words can **collide** and (b) the mapping **cannot be decoded**.

In [5]:
s = "The dog chased the cat!"
print("1000 buckets:", hash_encode(s, 1000))
print("unseen words work too:", hash_encode("zebras gallop quickly", 1000))

words = sorted({w.lower().strip(".,!?;") for t in corpus for w in t.split()})
for buckets in (1000, 50, 10):
    ids = [hash_encode(w, buckets)[0] for w in words]
    print(f"{buckets:>5} buckets: {len(words)} distinct words -> {len(set(ids))} distinct ids  (collisions: {len(words) - len(set(ids))})")

1000 buckets: [132, 168, 880, 132, 759, 285]
unseen words work too: [831, 161, 863]
 1000 buckets: 27 distinct words -> 26 distinct ids  (collisions: 1)
   50 buckets: 27 distinct words -> 18 distinct ids  (collisions: 9)
   10 buckets: 27 distinct words -> 8 distinct ids  (collisions: 19)


## 3. Byte-pair encoding (BPE) subwords
BPE repeatedly **merges the most frequent adjacent pair of symbols**, so frequent words become single tokens and rare / unseen words are split into familiar pieces
instead of becoming `<OOV>`. Encoding and decoding remain lossless.

In [6]:
bpe_corpus = ["low lower lowest", "new newer newest", "wide wider widest", "the lowest and widest"] * 3
bpe = BPE(num_merges=15).fit(bpe_corpus)
print("first merges learned:", [a + "+" + b for a, b in bpe.merges[:8]])
print()
for w in ["lowest", "newest", "wider", "lowest newest"]:
    print(f"{w!r:18} -> {bpe.tokenize(w)}")

first merges learned: ['w+e', 't+</w>', 's+t</w>', 'w+i', 'wi+d', 'wid+e', 'l+o', 'we+st</w>']

'lowest'           -> ['lowest</w>']
'newest'           -> ['ne', 'west</w>']
'wider'            -> ['wider</w>']
'lowest newest'    -> ['lowest</w>', 'ne', 'west</w>']


In [7]:
# An UNSEEN word: TextProcess can only say <OOV>, BPE still represents it with known pieces
unseen = "widest lower"
tp_demo = TextProcess().fit(["low lower lowest new newer newest wide wider"])
print("TextProcess ids :", tp_demo.encode("widest"), "->", tp_demo.decode(tp_demo.encode("widest")))
print("BPE pieces      :", bpe.tokenize("wester"))
ids = bpe.encode("wester lowest")
print("BPE ids         :", ids)
print("BPE decoded     :", repr(bpe.decode(ids)))
assert bpe.decode(ids) == "wester lowest"

TextProcess ids : [1] -> <OOV>
BPE pieces      : ['we', 's', 't', 'e', 'r</w>']
BPE ids         : [22, 16, 18, 5, 15, 10]
BPE decoded     : 'wester lowest'


## 4. From ids to embeddings: a tiny sentiment classifier
Integer ids are only labels. An **Embedding layer** turns each id into a trainable dense vector, and a small network can then learn from the text.
The data below are *template-generated toy sentences* (not real reviews) - the point is the pipeline
`text -> TextProcess ids -> Embedding -> prediction -> decode`, not the accuracy number.

In [8]:
texts, y = make_toy_sentiment(400, seed=0)
tp = TextProcess().fit(texts[:300])
print("examples:", [(t, int(l)) for t, l in zip(texts[:3], y[:3])])
print("vocabulary size:", tp.vocab_size)

model, hist = train_embedding_classifier(tp=tp, train_texts=texts[:300], y_train=y[:300], epochs=25)
X_test = tp.encode_batch(texts[300:], maxlen=8)
probs = model.predict(X_test, verbose=0).ravel()
acc = float(((probs >= 0.5) == y[300:]).mean())
print(f"held-out accuracy on {len(X_test)} sentences: {acc:.1%}")
assert acc > 0.9

examples: [('The service was dreadful.', 0), ('That service is annoying, honestly.', 0), ('Such a poor meal today.', 0)]
vocabulary size: 41


I0000 00:00:1795076825.185649    4288 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795076825.234311    4288 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1795076826.528756    4288 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


held-out accuracy on 100 sentences: 100.0%


In [9]:
for t in ["The movie was wonderful.", "What a terrible meal.", "The trip is boring, honestly."]:
    ids = tp.encode(t)
    p = float(model.predict(tp.encode_batch([t], maxlen=8), verbose=0)[0, 0])
    print(f"{t!r:34} ids={ids}  decoded={tp.decode(ids)!r}  P(positive)={p:.2f}")

'The movie was wonderful.'         ids=[3, 19, 4, 38, 2]  decoded='the movie was wonderful.'  P(positive)=1.00


'What a terrible meal.'            ids=[16, 5, 39, 22, 2]  decoded='what a terrible meal.'  P(positive)=0.00
'The trip is boring, honestly.'    ids=[3, 21, 7, 27, 8, 9, 2]  decoded='the trip is boring, honestly.'  P(positive)=0.00


In [10]:
# Limitation: a word never seen during fit becomes <OOV>, so the model has nothing to go on
t = "The movie was spectacular."
print(t, "->", tp.encode(t), "->", tp.decode(tp.encode(t)))
print("P(positive) =", round(float(model.predict(tp.encode_batch([t], maxlen=8), verbose=0)[0, 0]), 2),
      "(the sentiment word is <OOV>, so this is essentially a guess)")

The movie was spectacular. -> [3, 19, 4, 1, 2] -> the movie was <OOV>.


P(positive) = 0.5 (the sentiment word is <OOV>, so this is essentially a guess)


## 5. Summary
* **Bag-of-words / TF-IDF**: simple, strong baselines for classification, but order is lost and nothing can be decoded.
* **Hashing**: no vocabulary to store, robust to new words, but collisions are unavoidable and it is one-way.
* **BPE**: keeps the round trip lossless *and* removes most `<OOV>` problems - the idea behind the tokenizers of modern language models.
* **Embeddings**: the bridge from integer ids to neural networks; their quality depends on the tokenisation underneath.
* All functions are covered by `pytest` (`tests/test_text_extras.py`).